# WOD-E2E Minimal-Shot AV Analysis

This notebook is the analysis record for the Grand Commission WOD-E2E track. It is deliberately written as an exploration log, not just a final metric table.

Important split note: this workspace retained the WOD-E2E validation split and 479 preference-labeled validation frames, but the full local train/test download was not kept to save disk. Because of that, this notebook treats the retained validation preference frames as the analysis proxy and uses segment-grouped cross-validation to avoid frame-level leakage. The workflow below is written so the same inspection cells can be rerun when the full train/test shards are restored. No hidden-test or leaderboard result is claimed.

In [ ]:
from pathlib import Path
import json

ROOT = Path('..').resolve()
VAL_DIR = ROOT / 'waymo_open_dataset_end_to_end_camera_v_1_0_0' / 'val'
FRAME_CACHE = ROOT / 'artifacts' / 'wod_preference_frames_val479.json'
PROMOTED_REPORT = ROOT / 'artifacts' / 'wod_fastkin_gate_ridge175_rate020_fallback_cv_official.json'
BREAKTHROUGH_AUDIT = ROOT / 'artifacts' / 'wod_fastkin_gate_ridge175_rate020_fallback_breakthrough_audit.json'

def load_json(path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

print('validation dir exists:', VAL_DIR.exists())
print('validation shard count:', len(list(VAL_DIR.glob('*.tfrecord-*'))) if VAL_DIR.exists() else 0)
print('frame cache exists:', FRAME_CACHE.exists())
print('promoted report exists:', PROMOTED_REPORT.exists())
print('breakthrough audit exists:', BREAKTHROUGH_AUDIT.exists())

## 1. Dataset Access And Split Declaration

The intended deliverable asks for an analysis notebook on the training set. The full WOD-E2E download was not retained in this workspace, so the train and test shards need to be restored before a strict train-set notebook pass. The available local data is:

- `waymo_open_dataset_end_to_end_camera_v_1_0_0/val`: 93 validation TFRecord shards.
- `artifacts/wod_preference_frames_val479.json`: parsed cache of 479 validation frames with valid rater preference labels.

This means all reported WOD metrics below are **development evidence** over validation preference frames. They are useful for understanding design choices and failure modes, but they are not a leaderboard score and not strict zero-shot evidence.

In [ ]:
if FRAME_CACHE.exists():
    cache = load_json(FRAME_CACHE)
    frames = cache.get('frames', [])
    print('cache schema:', cache.get('schema'))
    print('frames:', len(frames))
    if frames:
        sample = frames[0]
        print('sample keys:', sorted(sample.keys())[:20])
        print('sample frame name:', sample.get('frame_name'))
        print('past points:', len(sample.get('past_trajectory', [])))
        print('future points:', len(sample.get('future_trajectory', [])))
        print('reference count:', len(sample.get('references', [])))
else:
    print('Frame cache is missing. Generate it with scripts/evaluate_wod_trajectory_model_cv.py --frame-cache ...')

## 2. What I Inspected First

The first pass focused on fields available without assuming maps, lidar, object labels, or external perception annotations: frame name, segment grouping key, 4 Hz past ego trajectory, 5 second future target where labels are present, route intent, initial speed, and rater preference trajectories/scores on validation frames.

The design constraint was that the model path should remain small and auditable, so I started with non-text trajectory candidates before adding any neural proposal experiments.

In [ ]:
if FRAME_CACHE.exists():
    from collections import Counter
    frames = load_json(FRAME_CACHE).get('frames', [])
    intents = Counter(str(frame.get('intent')) for frame in frames)
    ref_counts = Counter(len(frame.get('references', [])) for frame in frames)
    print('intent counts:', dict(intents))
    print('rater reference counts:', dict(ref_counts))
    speeds = []
    for frame in frames:
        past = frame.get('past_trajectory', [])
        if len(past) >= 2:
            dx = past[-1][0] - past[-2][0]
            dy = past[-1][1] - past[-2][1]
            speeds.append((dx * dx + dy * dy) ** 0.5 * 4.0)
    print('approx speed mean:', round(sum(speeds) / len(speeds), 3) if speeds else None)
    print('approx speed min/max:', (round(min(speeds), 3), round(max(speeds), 3)) if speeds else None)

## 3. Scenario Focus

The official scenario-cluster labels were not available as a local structured table, so I focused on behavior slices derived from parsed frames and aligned with the long-tail brief:

- stopped or creep: likely occlusion, waiting, and ambiguous right-of-way cases,
- slow / urban: dense interaction cases where constant velocity is often misleading,
- fast: route-following and trajectory-continuity mistakes are expensive,
- left / right intent: turn geometry exposes whether candidates respect route commands.

For the simulation side, scenario clusters are explicit and cover construction, intersections, pedestrians, cyclists, cut-ins, debris, special vehicles, spotlight hazards, and unusual maneuvers.

## 4. Candidate And Selector Scaffolds Tried

I tested the system as two separable problems: candidate diversity and candidate selection.

Candidate sources tried: constant velocity / acceleration / hold / heading-change kinematics, ridge residual trajectory models, temporal-summary auxiliary residual models, external-embedding scene auxiliary candidates, lightweight neural and transformer proposal experiments, and memory/world-model candidates.

Selector scaffolds tried: absolute RFS regression, frame-delta targets, normalized frame-delta targets, pairwise and listwise rankers, source calibration, fallback routing, and scene/source gates.

The main lesson was that proposal oracle headroom was easier to create than selector reliability. Many candidate additions improved oracle RFS but hurt selected RFS.

In [ ]:
if PROMOTED_REPORT.exists():
    report = load_json(PROMOTED_REPORT)
    keys = [
        'score_backend', 'frames', 'fold_count',
        'kinematic_constant_velocity_mean_rfs',
        'combined_ranker_mean_rfs',
        'combined_oracle_mean_rfs',
        'combined_ranker_regret_to_oracle',
        'selected_kinematic_rate', 'selected_temporal_rate',
        'selected_learned_rate', 'selected_scene_rate',
    ]
    for key in keys:
        print(f'{key}:', report.get(key))
else:
    print('Promoted report missing')

## 5. What Worked

The best confirmed WOD development result is the fallback-enabled fast-kinematic/scene-gate structured selector report:

- constant velocity official validation-CV RFS: `7.022`,
- selected combined candidate RFS: `7.657`,
- gain over constant velocity: `+0.635`,
- combined candidate oracle: `9.098`,
- selector regret to oracle: `1.441`.

What survived into the final system: simple kinematic candidates, residual ridge candidates, temporal auxiliary candidates, conservative fallback/gating, and explicit audits separating validation-CV evidence from hidden-test truth.

In [ ]:
if PROMOTED_REPORT.exists():
    report = load_json(PROMOTED_REPORT)
    slices = report.get('slices', {})
    rows = []
    for name, payload in slices.items():
        rows.append((name, payload.get('frames'), payload.get('selected_mean_rfs'), payload.get('oracle_mean_rfs'), payload.get('mean_regret')))
    for row in sorted(rows, key=lambda item: item[-1] if item[-1] is not None else -1, reverse=True):
        print(row)

## 6. What Failed

The promoted WOD validation report is **not** a clean breakthrough by the repo's own audit gate. Mean RFS improved, but worst-slice regret regressed.

Rejected or de-emphasized paths:

- lightweight world-model candidates produced only a tiny selected-RFS movement (`7.6028` to `7.6062`),
- neural proposal candidates improved held-out slices but did not transfer cleanly to the full 479-frame official run,
- urban kinematic and learned-source gates regressed in follow-up official scorer checks,
- many proposal additions increased oracle headroom without improving the deployed selector.

This failure shaped the final story: the submission should emphasize architecture, closed-loop behavior, audits, and candidate-selection bottlenecks rather than claiming solved WOD-E2E autonomy.

In [ ]:
if BREAKTHROUGH_AUDIT.exists():
    audit = load_json(BREAKTHROUGH_AUDIT)
    print('passed:', audit.get('passed'))
    print('failures:')
    for failure in audit.get('failures', []):
        print('-', failure)
    print('metrics:', audit.get('metrics'))
else:
    print('Breakthrough audit missing')

## 7. How Analysis Shaped The Final System

The analysis changed the system in four concrete ways:

1. No hidden-test claim: readiness audits fail when test data or leaderboard logs are missing.
2. Candidate diversity is separated from selection: oracle headroom is reported separately from selected RFS.
3. Fallback/gating is explicit: the best WOD development result comes from conservative routing around selector weaknesses.
4. Simulation is a separate track: closed-loop Spotlight Reflex evidence is not used to tune WOD RFS.

Final Grand claim: a runnable minimal-shot autonomy architecture prototype with WOD-E2E harness, reproducible validation-CV evidence, closed-loop demos, and honest failure analysis.

## 8. Next Work If Train/Test Splits Become Available

When the official train and test shards are restored locally, the next analysis pass should run the same field inspection on train shards, extract train-set anchors/residuals without validation labels, reserve validation preference frames for calibration and failure analysis, generate a pre-registered blind test submission matrix, record hidden-test leaderboard results by exact SHA-256, and update this notebook with train-set montage examples and train-cluster statistics.

Until then, this notebook remains a validation-proxy analysis record.